# Alerts as data: investigating alert storms with ES|QL

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/alerts-as-data-log-derived-signals-v2). Run the cells in order: they build the lab, replay two incidents, run the investigation queries, register the agent, and delete everything again at the end.

Before you start: Elasticsearch and Kibana 9.5, Agent Builder enabled for the last section, and a `.env` copied from `env.example`. Runs for about 25 minutes, because the rules evaluate once a minute and the episode durations are the data.


In [ ]:
%pip install -q elasticsearch==9.5.0 python-dotenv==1.0.1 requests==2.32.3

## 1. Connect

`KIBANA_SPACE` must name a space created with the Observability solution; the Alerts page this article uses does not exist in an Elasticsearch-solution space.


In [ ]:
import json
import os
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

load_dotenv()

BASE = os.getenv("KIBANA_URL").rstrip("/")
KIBANA_SPACE = os.getenv("KIBANA_SPACE", "default")
if KIBANA_SPACE != "default":
    BASE = f"{BASE}/s/{KIBANA_SPACE}"

API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
es = Elasticsearch(os.getenv("ELASTICSEARCH_URL"), api_key=API_KEY, request_timeout=60)
HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, payload=None):
    """Call a Kibana API and raise with the response body on failure."""
    response = requests.request(
        method,
        f"{BASE}{path}",
        headers=HEADERS,
        data=json.dumps(payload) if payload is not None else None,
        timeout=60,
    )
    if not response.ok:
        raise RuntimeError(
            f"{method} {path} -> {response.status_code}: {response.text}"
        )
    return response.json() if response.text else {}


# Every rule carries this tag. It lands on each alert document as
# kibana.alert.rule.tags, which is how every query below isolates the lab.
LAB_TAG = "alerts-as-data-v2"
DATA_STREAM = "logs-payments.stormlab-default"
INDEX_PATTERN = "logs-payments.stormlab*"
CATALOG_INDEX = "payments_service_catalog"
REGION = "us-central1"

RULE_GATEWAY = "gateway-5xx-per-endpoint-status"
RULE_QUEUE_LAG = "fraud-scorer-queue-lag"
RULE_ERROR_RATE = "payments-error-rate-per-service"

print(es.info()["version"]["number"])

## 2. Service catalog

`index.mode: lookup` is what allows `LOOKUP JOIN` from ES|QL later.


In [ ]:
es.options(ignore_status=404).indices.delete(index=CATALOG_INDEX)

es.indices.create(
    index=CATALOG_INDEX,
    settings={"index.mode": "lookup"},
    mappings={
        "properties": {
            "service.name": {"type": "keyword"},
            "team": {"type": "keyword"},
            "tier": {"type": "keyword"},
            "depends_on": {"type": "keyword"},
        }
    },
)

catalog_docs = [
    {
        "service.name": "fraud-scorer",
        "team": "risk-ml",
        "tier": "backend",
        "depends_on": "payments-queue",
    },
    {
        "service.name": "payment-gateway",
        "team": "payments-core",
        "tier": "edge",
        "depends_on": "fraud-scorer",
    },
    {
        "service.name": "payments-web",
        "team": "storefront",
        "tier": "frontend",
        "depends_on": "payment-gateway",
    },
]

helpers.bulk(
    es, ({"_index": CATALOG_INDEX, "_source": d} for d in catalog_docs), refresh=True
)
print(f"{len(catalog_docs)} catalog documents indexed")

## 3. Log data stream

Same field shape the OTel Collector produces: `labels.*` for strings, `numeric_labels.*` for numbers.


In [ ]:
es.indices.put_index_template(
    name="logs-payments-stormlab",
    index_patterns=[INDEX_PATTERN],
    data_stream={},
    priority=500,
    template={
        "mappings": {
            "properties": {
                "@timestamp": {"type": "date"},
                "message": {"type": "text"},
                "service.name": {"type": "keyword"},
                "service.version": {"type": "keyword"},
                "log.level": {"type": "keyword"},
                "cloud.region": {"type": "keyword"},
                "labels": {
                    "properties": {
                        "endpoint": {"type": "keyword"},
                        "status_code": {"type": "keyword"},
                        "incident_id": {"type": "keyword"},
                    }
                },
                "numeric_labels": {
                    "properties": {
                        "queue_lag_seconds": {"type": "double"},
                        "latency_ms": {"type": "double"},
                    }
                },
            }
        }
    },
)

print("index template created")

## 4. Incident generator

Seven `(endpoint, status)` pairs are what make the miscalibrated rule expensive: grouping by both turns each pair into its own alert instance.


In [ ]:
GATEWAY_PAIRS = [
    ("/api/payments", "503"),
    ("/api/wallet/topup", "500"),
    ("/api/payments/confirm", "504"),
    ("/api/wallet/topup", "503"),
    ("/api/refunds", "500"),
    ("/api/payments/confirm", "503"),
    ("/api/refunds", "504"),
]

GATEWAY_START_MINUTE = 5
WEB_START_MINUTE = 6


def log_doc(service, level, message, incident_id, **fields):
    doc = {
        "@timestamp": datetime.now(timezone.utc).isoformat(),
        "service.name": service,
        "log.level": level,
        "message": message,
        "cloud.region": REGION,
        "labels": {"incident_id": incident_id},
        "numeric_labels": {},
    }
    for key, value in fields.items():
        if key in ("endpoint", "status_code"):
            doc["labels"][key] = value
        elif key in ("queue_lag_seconds", "latency_ms"):
            doc["numeric_labels"][key] = value
        else:
            doc[key] = value
    return doc


def write(docs):
    # Data streams only accept op_type "create"; helpers.bulk defaults to "index".
    if docs:
        helpers.bulk(
            es,
            ({"_index": DATA_STREAM, "_op_type": "create", "_source": d} for d in docs),
            refresh=True,
        )
    return len(docs)

## 5. Replay

One batch per minute: healthy baseline, the deploy, queue lag past the threshold, rotating gateway 5xx bursts, customer-visible errors, rollback.


In [ ]:
def build_minute(minute, incident_id, storm_minutes):
    """Return one replay minute with a deliberately staggered cascade.

    minute 0-2  : healthy baseline
    minute 3    : deploy symptom, one fraud-scorer error below the lag threshold
    minute 4    : fraud-scorer crosses the queue-lag threshold
    minute 5..N : gateway emits seven 5xx logs across two rotating pairs
    minute 6..N : customers see errors on payments-web
    last minute : rollback, everything recovers
    """
    docs = []
    storm_end = GATEWAY_START_MINUTE + storm_minutes

    for _ in range(4):
        docs.append(
            log_doc(
                "payment-gateway",
                "INFO",
                "checkout completed",
                incident_id,
                endpoint="/api/payments",
                status_code="200",
                latency_ms=120,
            )
        )

    if minute < 3:
        docs.append(
            log_doc(
                "fraud-scorer",
                "INFO",
                "model scored batch",
                incident_id,
                queue_lag_seconds=2.0,
            )
        )
        return docs

    if minute == 3:
        docs.append(
            log_doc(
                "fraud-scorer",
                "ERROR",
                "slow model load after deploy",
                incident_id,
                queue_lag_seconds=15.0,
            )
        )
        return docs

    if minute >= storm_end:
        docs.append(
            log_doc(
                "fraud-scorer",
                "INFO",
                "rolled back to 2.2.9, queue draining",
                incident_id,
                queue_lag_seconds=3.0,
            )
        )
        return docs

    # fraud-scorer: slow model loading, queue backs up
    lag = 30.0 + (minute - 4) * 14.0
    docs.append(
        log_doc(
            "fraud-scorer",
            "ERROR",
            f"scoring backlog, consumer lag {lag:.0f}s",
            incident_id,
            queue_lag_seconds=lag,
        )
    )
    for _ in range(2):
        docs.append(
            log_doc(
                "fraud-scorer",
                "ERROR",
                "model load timeout",
                incident_id,
                queue_lag_seconds=lag,
            )
        )

    # payment-gateway: seven errors/minute, split 4+3 across two rotating pairs
    if minute >= GATEWAY_START_MINUTE:
        offset = ((minute - GATEWAY_START_MINUTE) * 2) % len(GATEWAY_PAIRS)
        pair_indexes = (offset, (offset + 1) % len(GATEWAY_PAIRS))
        for pair_index, repeats in zip(pair_indexes, (4, 3)):
            endpoint, status = GATEWAY_PAIRS[pair_index]
            for _ in range(repeats):
                docs.append(
                    log_doc(
                        "payment-gateway",
                        "ERROR",
                        f"upstream timeout on {endpoint}",
                        incident_id,
                        endpoint=endpoint,
                        status_code=status,
                        latency_ms=9000,
                    )
                )

    # payments-web: customers notice a minute later
    if minute >= WEB_START_MINUTE:
        for _ in range(4):
            docs.append(
                log_doc(
                    "payments-web",
                    "ERROR",
                    "checkout failed for customer",
                    incident_id,
                    endpoint="/checkout",
                    status_code="502",
                )
            )

    return docs


def replay_incident(incident_id, storm_minutes, total_minutes, before_minute=None):
    print(f"replaying {incident_id}: {total_minutes} minutes")
    for minute in range(total_minutes):
        if before_minute:
            before_minute(minute)
        started = time.monotonic()
        count = write(build_minute(minute, incident_id, storm_minutes))
        print(
            f"  minute {minute:>2}  {count:>3} docs  {datetime.now().strftime('%H:%M:%S')}"
        )
        if minute < total_minutes - 1:
            time.sleep(max(0, 60 - (time.monotonic() - started)))
    print("replay complete")


def wait_for_alerts(total, timeout=420):
    """Block until every episode has opened and recovered, so the counts are final."""
    deadline = time.monotonic() + timeout
    previous = None
    while time.monotonic() < deadline:
        response = es.search(
            index=".alerts-*",
            size=0,
            track_total_hits=True,
            query={"term": {"kibana.alert.rule.tags": LAB_TAG}},
            aggs={"statuses": {"terms": {"field": "kibana.alert.status", "size": 10}}},
        )
        statuses = {
            b["key"]: b["doc_count"]
            for b in response["aggregations"]["statuses"]["buckets"]
        }
        state = (response["hits"]["total"]["value"], statuses.get("active", 0))
        if state != previous:
            print(f"  {state[0]} episodes, {state[1]} active")
            previous = state
        if state == (total, 0):
            return
        time.sleep(5)
    print(
        f"!! timed out waiting for {total} recovered episodes; last state was {previous}"
    )

## 6. Data view

Needed by the custom threshold rule only.


In [ ]:
existing = kbn("GET", "/api/data_views")
data_view_id = next(
    (
        dv["id"]
        for dv in existing.get("data_view", [])
        if dv.get("title") == INDEX_PATTERN
    ),
    None,
)

if data_view_id is None:
    created = kbn(
        "POST",
        "/api/data_views/data_view",
        {
            "data_view": {
                "title": INDEX_PATTERN,
                "name": "payments stormlab logs",
                "timeFieldName": "@timestamp",
            }
        },
    )
    data_view_id = created["data_view"]["id"]

print("data view:", data_view_id)

## 7. The three rules

Rule 1 is deliberately miscalibrated, grouped by endpoint **and** status code with a one-minute window; rule 2 watches queue lag; rule 3 is the sane default, grouped by service.


In [ ]:
gateway_esql_rev0 = (
    f"FROM {INDEX_PATTERN} "
    '| WHERE service.name == "payment-gateway" AND log.level == "ERROR" '
    "| STATS error_count = COUNT(*) BY labels.endpoint, labels.status_code, cloud.region "
    "| WHERE error_count >= 3"
)

queue_lag_esql = (
    f"FROM {INDEX_PATTERN} "
    '| WHERE service.name == "fraud-scorer" AND numeric_labels.queue_lag_seconds IS NOT NULL '
    "| STATS max_lag_seconds = MAX(numeric_labels.queue_lag_seconds) BY service.name, cloud.region "
    "| WHERE max_lag_seconds >= 30"
)


def esql_params(esql, window_minutes):
    return {
        "searchType": "esqlQuery",
        "esqlQuery": {"esql": esql},
        "timeField": "@timestamp",
        "timeWindowSize": window_minutes,
        "timeWindowUnit": "m",
        "thresholdComparator": ">",
        "threshold": [0],
        "size": 0,
        "excludeHitsFromPreviousRun": False,
        "aggType": "count",
        "groupBy": "row",  # one alert per returned row, not one for the whole result
    }


threshold_params = {
    "criteria": [
        {
            "comparator": ">",
            "threshold": [5],
            "timeSize": 2,
            "timeUnit": "m",
            "metrics": [
                {"name": "A", "aggType": "count", "filter": "log.level: ERROR"}
            ],
        }
    ],
    "groupBy": ["service.name"],
    "alertOnNoData": False,
    "alertOnGroupDisappear": False,
    "searchConfiguration": {
        "index": data_view_id,
        "query": {"query": "", "language": "kuery"},
    },
}


def authorized_consumer(rule_type_id, params):
    """First consumer this API key can create the rule under AND list back.

    Creating is not enough: a rule created under a consumer the key cannot list
    fires correctly but stays invisible in the Kibana interface.
    """
    for candidate in ("alerts", "stackAlerts", "observability"):
        probe = None
        try:
            probe = kbn(
                "POST",
                "/api/alerting/rule",
                {
                    "name": f"__consumer_probe_{candidate}",
                    "rule_type_id": rule_type_id,
                    "consumer": candidate,
                    "enabled": False,
                    "tags": ["__consumer_probe"],
                    "schedule": {"interval": "1m"},
                    "actions": [],
                    "params": params,
                },
            )["id"]
            found = kbn(
                "GET",
                "/api/alerting/rules/_find?per_page=100&search_fields=tags&search=__consumer_probe",
            )
            if any(r["id"] == probe for r in found.get("data", [])):
                return candidate
        except RuntimeError:
            pass
        finally:
            if probe:
                kbn("DELETE", f"/api/alerting/rule/{probe}")
    raise RuntimeError(
        f"No consumer works for {rule_type_id}; check the Kibana Alerting privileges."
    )


def create_rule(name, rule_type_id, consumer, params):
    return kbn(
        "POST",
        "/api/alerting/rule",
        {
            "name": name,
            "rule_type_id": rule_type_id,
            "consumer": consumer,
            "enabled": True,
            "tags": [LAB_TAG],
            "schedule": {"interval": "1m"},
            "actions": [],
            "params": params,
        },
    )


esql_consumer = authorized_consumer(".es-query", esql_params(queue_lag_esql, 2))
threshold_consumer = authorized_consumer(
    "observability.rules.custom_threshold", threshold_params
)

# Create the queue rule first so its scheduler observes the upstream symptom first.
queue_lag_rule = create_rule(
    RULE_QUEUE_LAG, ".es-query", esql_consumer, esql_params(queue_lag_esql, 2)
)
gateway_rule = create_rule(
    RULE_GATEWAY, ".es-query", esql_consumer, esql_params(gateway_esql_rev0, 1)
)
threshold_rule = create_rule(
    RULE_ERROR_RATE,
    "observability.rules.custom_threshold",
    threshold_consumer,
    threshold_params,
)

print("three rules created")

## 8. First incident

`fraud-scorer` 2.3.0 loads slowly, the queue backs up, the gateway times out, customers see failed checkouts. Runs for about 13 minutes.


In [ ]:
INCIDENT_1 = "payments-brownout-20260722"

replay_incident(INCIDENT_1, storm_minutes=7, total_minutes=13)
wait_for_alerts(total=18)

## 9. Investigation queries

Only alert history from here on. Expect 18 episodes from 11 instances, `fraud-scorer` first, and a median of one minute for the gateway rule.


In [ ]:
def show(query, title):
    response = es.esql.query(query=query, format="json")
    print(f"\n{title}")
    print("-" * len(title))
    print(" | ".join(c["name"] for c in response["columns"]))
    for row in response["values"]:
        print(" | ".join("" if v is None else str(v) for v in row))
    return response["values"]


q1 = f"""
FROM .alerts-*
| WHERE kibana.alert.rule.tags == "{LAB_TAG}"
| STATS episodes = COUNT(*),
        alert_instances = COUNT_DISTINCT(kibana.alert.instance.id),
        first_episode = MIN(kibana.alert.start),
        last_episode = MAX(kibana.alert.start)
  BY rule = kibana.alert.rule.name
| SORT episodes DESC
"""

show(q1, "Question 1: one incident or many?")

In [ ]:
q2 = f"""
FROM .alerts-*
| WHERE kibana.alert.rule.tags == "{LAB_TAG}" AND service.name IS NOT NULL
| STATS first_symptom = MIN(kibana.alert.start) BY service.name
| LOOKUP JOIN {CATALOG_INDEX} ON service.name
| KEEP first_symptom, service.name, team, tier, depends_on
| SORT first_symptom
"""

show(q2, "Question 2: where did it start, and whose problem is it?")

In [ ]:
q3 = f"""
FROM .alerts-*
| WHERE kibana.alert.rule.tags == "{LAB_TAG}"
| EVAL minutes_active = COALESCE(kibana.alert.duration.us, 0) / 60000000.0
| STATS episodes = COUNT(*),
        alert_instances = COUNT_DISTINCT(kibana.alert.instance.id),
        flapping_episodes = SUM(CASE(kibana.alert.flapping == true, 1, 0)),
        median_minutes_active = MEDIAN(minutes_active)
  BY rule = kibana.alert.rule.name, revision = kibana.alert.rule.revision
| SORT episodes DESC
"""

show(q3, "Question 3: which rule needs fixing, and by how much?")

## 10. The fix

Update the rule instead of replacing it: Kibana stamps `kibana.alert.rule.revision`, so before and after stay separable in the same index.


In [ ]:
gateway_esql_rev1 = (
    f"FROM {INDEX_PATTERN} "
    '| WHERE service.name == "payment-gateway" AND log.level == "ERROR" '
    "| STATS error_count = COUNT(*) BY service.name, cloud.region "
    "| WHERE error_count >= 10"
)

# Keep the rule disabled until its new 5m window can no longer see incident 1.
kbn("POST", f"/api/alerting/rule/{gateway_rule['id']}/_disable")
kbn(
    "PUT",
    f"/api/alerting/rule/{gateway_rule['id']}",
    {
        "name": RULE_GATEWAY,
        "tags": [LAB_TAG],
        "schedule": {"interval": "1m"},
        "actions": [],
        "params": esql_params(gateway_esql_rev1, 5),
    },
)

updated = kbn("GET", f"/api/alerting/rule/{gateway_rule['id']}")
print("revision:", updated["revision"], "| enabled:", updated["enabled"])

## 11. Second incident

Same failure shape, compressed. Runs for about 11 minutes.


In [ ]:
INCIDENT_2 = "payments-brownout-20260722-b"


def wait_until_gateway_window_clear(timeout=300):
    """Revision 1 looks back five minutes; it must not see incident 1's errors."""
    deadline = time.monotonic() + timeout
    query = {
        "bool": {
            "filter": [
                {"term": {"service.name": "payment-gateway"}},
                {"term": {"log.level": "ERROR"}},
                {"range": {"@timestamp": {"gte": "now-5m"}}},
            ]
        }
    }
    while time.monotonic() < deadline:
        recent = es.count(index=INDEX_PATTERN, query=query)["count"]
        if recent == 0:
            print("verified: revision 1 window contains no revision 0 gateway errors")
            return
        print(
            f"  waiting for {recent} old gateway error log(s) to leave the 5m window..."
        )
        time.sleep(15)
    raise RuntimeError(
        "Old gateway errors are still inside the revision 1 lookback window."
    )


def enable_gateway_rule(minute):
    if minute != 4:
        return
    wait_until_gateway_window_clear()
    kbn("POST", f"/api/alerting/rule/{gateway_rule['id']}/_enable")
    print("gateway revision 1 enabled after a clean 5m lookback")


replay_incident(
    INCIDENT_2, storm_minutes=5, total_minutes=11, before_minute=enable_gateway_rule
)
wait_for_alerts(total=23)

## 12. Scorecard after the fix

The same query, and now the revision column earns its place: fourteen episodes at revision 0 against one at revision 1.


In [ ]:
show(q3, "Scorecard after the fix, split by revision")

## 13. Agent Builder

Each tool is a parameterized version of one query above. The `KEEP` projection matters: the alert indices map around 1,400 fields.


In [ ]:
AGENT_ID = "alert_historian"
LOOKBACK = {
    "lookback_hours": {
        "type": "integer",
        "description": "How many hours of alert history to analyze, for example 6",
        "optional": True,
        "defaultValue": 6,
    }
}
WINDOW = (
    f'WHERE kibana.alert.rule.tags == "{LAB_TAG}" '
    'AND DATE_DIFF("hours", kibana.alert.start, NOW()) <= ?lookback_hours'
)

TOOLS = [
    {
        "id": "alert_episode_timeline",
        "type": "esql",
        "description": "Chronological list of individual alert episodes. Use the gaps between episode starts to separate incident storms.",
        "configuration": {
            "query": (
                f"FROM .alerts-* | {WINDOW} "
                "| KEEP kibana.alert.start, kibana.alert.end, kibana.alert.rule.name, "
                "kibana.alert.rule.revision, kibana.alert.instance.id, service.name "
                "| SORT kibana.alert.start | LIMIT 100"
            ),
            "params": LOOKBACK,
        },
    },
    {
        "id": "alert_first_symptom_by_service",
        "type": "esql",
        "description": "First alerting symptom per service, joined against the service catalog for owning team, tier, and upstream dependency.",
        "configuration": {
            "query": (
                f"FROM .alerts-* | {WINDOW} AND service.name IS NOT NULL "
                "| STATS first_symptom = MIN(kibana.alert.start) BY service.name "
                f"| LOOKUP JOIN {CATALOG_INDEX} ON service.name "
                "| KEEP first_symptom, service.name, team, tier, depends_on | SORT first_symptom | LIMIT 10"
            ),
            "params": LOOKBACK,
        },
    },
    {
        "id": "alert_noise_scorecard",
        "type": "esql",
        "description": "Noise scorecard per alerting rule and revision: episodes, distinct instances, flapping episodes, median minutes active.",
        "configuration": {
            "query": (
                f"FROM .alerts-* | {WINDOW} "
                "| EVAL minutes_active = COALESCE(kibana.alert.duration.us, 0) / 60000000.0 "
                "| STATS episodes = COUNT(*), alert_instances = COUNT_DISTINCT(kibana.alert.instance.id), "
                "flapping_episodes = SUM(CASE(kibana.alert.flapping == true, 1, 0)), "
                "median_minutes_active = MEDIAN(minutes_active) "
                "BY rule = kibana.alert.rule.name, revision = kibana.alert.rule.revision "
                "| KEEP rule, revision, episodes, alert_instances, flapping_episodes, median_minutes_active "
                "| SORT episodes DESC | LIMIT 10"
            ),
            "params": LOOKBACK,
        },
    },
]

for tool in TOOLS:
    kbn("POST", "/api/agent_builder/tools", tool)
    kbn(
        "POST",
        "/api/agent_builder/tools/_execute",
        {"tool_id": tool["id"], "tool_params": {"lookback_hours": 3}},
    )
    print("created and tested tool:", tool["id"])

## 14. The agent

The instructions define the vocabulary, the method, and the honesty rules: report numbers from tool output, never invent alerts.


In [ ]:
INSTRUCTIONS = """You are Alert Historian. You analyze preserved Elastic alert history.

Vocabulary:
- An "episode" is one alert document: one instance from the moment it became active until it recovered.
- An "instance" is one grouping-key value. The same instance can produce many episodes.

Method, in this order:
1. Call alert_episode_timeline first. It returns individual episode starts in order. Treat a gap of at least three minutes with no new episode as a boundary between storms. Count the returned rows for the total number of episodes.
2. Call alert_first_symptom_by_service. Report the earliest service, its owning team, and the propagation order. If the earliest alerting service depends on something that never alerted, say so - the true origin is probably one hop upstream.
3. Call alert_noise_scorecard. Compare revisions of the same rule when more than one is present.

Honesty rules:
- Report only numbers and timestamps that appear in tool output. Never invent alerts.
- Timestamps in tool output are UTC. Say so.
- If a tool returns no rows, say the history is empty for that window rather than guessing."""

agent = kbn(
    "POST",
    "/api/agent_builder/agents",
    {
        "id": AGENT_ID,
        "name": "Alert Historian",
        "description": "Investigates preserved Elastic alert history with ES|QL.",
        "configuration": {
            "instructions": INSTRUCTIONS,
            "tools": [{"tool_ids": [t["id"] for t in TOOLS]}],
        },
    },
)

print("agent created:", agent.get("id", AGENT_ID))

Ask it in Kibana under **Agent Builder > Alert Historian**:

> Summarize what happened to the payments platform in the last 3 hours. Was it one incident or many? Which team owns the first symptom? And did the tuning change to the gateway rule actually reduce noise in the second failure?


## 15. Clean up

Deletes only what this notebook created. Run it before re-running the notebook: leftover rules would double every count.


In [ ]:
for rule in (gateway_rule, queue_lag_rule, threshold_rule):
    kbn("DELETE", f"/api/alerting/rule/{rule['id']}")

kbn("DELETE", f"/api/agent_builder/agents/{AGENT_ID}")
for tool in TOOLS:
    kbn("DELETE", f"/api/agent_builder/tools/{tool['id']}")

es.options(ignore_status=404).delete_by_query(
    index=".internal.alerts-*",
    query={"term": {"kibana.alert.rule.tags": LAB_TAG}},
    refresh=True,
)
es.options(ignore_status=404).indices.delete_data_stream(name=DATA_STREAM)
es.options(ignore_status=404).indices.delete_index_template(
    name="logs-payments-stormlab"
)
es.options(ignore_status=404).indices.delete(index=CATALOG_INDEX)

print("cleanup complete")